In [ ]:
import pandas as pd
import numpy as np

squirrel = pd.read_csv("data/squirrel.csv")
hectare = pd.read_csv("data/hectare.csv", keep_default_na=False, na_values=[""]) # modified by HY, 260508

print("Squirrel:", squirrel.shape)
print("Hectare:", hectare.shape)

# squirrel.head()
# hectare.head()
# squirrel.info()

Squirrel: (3023, 31)
Hectare: (700, 13)


In [2]:
df = squirrel.merge(
    hectare,
    on=["Hectare", "Shift", "Date"],
    how="left"
)

print("After merge:", df.shape)


# df.isna().sum().sort_values(ascending=False).head(10)

# df

After merge: (3023, 41)


In [3]:
df["Unique Squirrel ID"].duplicated().sum()
# 5 duplicate squirrel id

dup = df[df["Unique Squirrel ID"].duplicated(keep=False)]
dup.sort_values("Unique Squirrel ID")
# only small differences occurs in x and y

df = df.drop_duplicates(
    subset=["Unique Squirrel ID"],
    keep="first"
)


df["Unique Squirrel ID"].duplicated().sum()

np.int64(0)

In [4]:
bool_cols = [
    "Running","Chasing","Climbing","Eating","Foraging",
    "Kuks","Quaas","Moans","Tail flags","Tail twitches",
    "Approaches","Indifferent","Runs from"
]


# df[bool_cols].apply(lambda x: x.unique())

In [5]:
# cell 5
age_mode = df["Age"].mode()
df["Age"] = df["Age"].replace("?", pd.NA)
df["Age"] = df["Age"].fillna(age_mode.iloc[0])


# df["Age"].unique()

# check the original data
# raw = pd.read_csv("a1-datasets/squirrel.csv")
# (raw["Age"] == "?").sum()
# raw["Age"].value_counts(dropna=False)


In [6]:
df["Date"] = pd.to_datetime(df["Date"], format="%m%d%Y", errors="coerce")
df["is_weekend"] = df["Date"].dt.dayofweek >= 5


# df[["Date","is_weekend"]].head()
# df["is_weekend"].value_counts(dropna=False)

In [7]:
df["Hectare Conditions"] = df["Hectare Conditions"].replace(
    {"Medium": "Moderate"}
)

df["Hectare Conditions"] = df["Hectare Conditions"].fillna(
    df["Hectare Conditions"].mode()[0]
)


#df["Hectare Conditions"].value_counts(dropna=False)

In [8]:
df["is_above_ground"] = df["Location"] == "Above Ground" # turn the data into boolean type

df["above_ground_numeric"] = pd.to_numeric(
    df["Above Ground Sighter Measurement"],
    errors="coerce"
)

df.loc[df["Location"] == "Ground Plane", "above_ground_numeric"] = 0 # set the ground level numerical

df["location_missing"] = df["Location"].isna()


# df[["Location","above_ground_numeric"]].head(10)
# df["above_ground_numeric"].describe()

In [9]:
colors = ["Gray", "White", "Cinnamon", "Black"]

for c in colors:
    df[f"highlight_{c.lower()}"] = df["Highlight Fur Color"].str.contains(
        c, case=False, na=False
    )

df["highlight_missing"] = df["Highlight Fur Color"].isna()


# df.filter(like="highlight_").sum()

In [10]:
keywords = ["human", "dog", "pigeon", "bird", "sparrow", "duck"]
# can change elements later, check the number using the code in the comment

for k in keywords:
    df[f"animals_{k}_present"] = df["Other Animal Sightings"].str.contains(
        k, case=False, na=False
    )

df["animals_data_missing"] = df["Other Animal Sightings"].isna()


animal_cols = [f"animals_{k}_present" for k in keywords] + ["animals_data_missing"]

# df[animal_cols].sum()

In [11]:
# cell 10
weather = df["Sighter Observed Weather Data"]

df["temp_unit"] = weather.str.extract(r"(F|C)", expand=False)
df["temperature"] = weather.str.extract(r"(\d+)").astype(float)

mask = df["temp_unit"] == "C"

df.loc[mask, "temperature"] = (
    df.loc[mask, "temperature"] * 9/5 + 32
)

df["temperature"] = df["temperature"].fillna(
    df["temperature"].median()
)

df = df.drop(columns=["temp_unit"]) # temperature already has the same unit

df["sky_condition"] = weather.str.extract(
    r"(?i)(sunny|cloudy|clouds|rainy|rain|drizzle|sprinkling|overcast|clear|fog|mist|wet)",
    expand=False
).str.lower()
# ignores the upper and lower case, and only takes the first word in string

df["sky_condition"] = df["sky_condition"].replace({
    "clouds": "cloudy",
    "rain": "rainy"
})

# df[["temperature_f","sky_condition"]].head()

In [12]:
df["Total Time of Sighting"] = df["Total Time of Sighting"].fillna(
    df["Total Time of Sighting"].median()
)


# df["Total Time of Sighting"].describe()

In [13]:
df["squirrel_density_proxy"] = (
    df["Number of Squirrels"] / df["Total Time of Sighting"].replace(0, np.nan)
)

df["squirrel_density_proxy"] = df["squirrel_density_proxy"].fillna(
    df["squirrel_density_proxy"].median()
)


# df["squirrel_density_proxy"].describe()

In [14]:
df["Primary Fur Color"] = df["Primary Fur Color"].fillna(
    df["Primary Fur Color"].mode()[0]
)

df["Primary Fur Color"] = df["Primary Fur Color"].fillna(
    df["Primary Fur Color"].mode()[0]
)


# df["Primary Fur Color"].value_counts(dropna=False)

In [15]:
df["Number of sighters"] = df["Number of sighters"].fillna(
    df["Number of sighters"].median()
)

df["Number of Squirrels"] = df["Number of Squirrels"].fillna(
    df["Number of Squirrels"].median()
)

In [16]:
# optimal part

df["session_id"] = (
    df["Hectare"].astype(str) + "_" +
    df["Shift"].astype(str) + "_" +
    df["Date"].dt.strftime("%Y-%m-%d")
)

# df["session_id"].head()

In [17]:
# check the number of session
# df["session_id"].nunique()

In [ ]:
drop_cols = [
    # raw columns replaced
    "Location",
    "Above Ground Sighter Measurement",
    "Highlight Fur Color",
    "Other Animal Sightings",
    # "Sighter Observed Weather Data",

    # text / noisy
    "Color notes",
    "Other Activities",
    "Specific Location",
    "Other Interactions",
    "Litter Notes",
    "Hectare Conditions Notes",

    # leakage
    # "Approaches",
    # "Indifferent",
    # "Runs from",

    # useless
    "Combination of Primary and Highlight Color",
    "Lat/Long",
    "Hectare Squirrel Number",
    "Anonymized Sighter",

    # avoid overfitting(optimal part)
    # "session_id"
] # modified by HY, 260508

df = df.drop(columns=drop_cols, errors="ignore")


# df.columns

In [19]:
print("Final shape:", df.shape)
# print(df["y"].value_counts())

df.isna().sum().sort_values(ascending=False).head(10)


Final shape: (3018, 47)


sky_condition                    520
Litter                           382
above_ground_numeric             114
Sighter Observed Weather Data     92
X                                  0
Y                                  0
Unique Squirrel ID                 0
Primary Fur Color                  0
Running                            0
Chasing                            0
dtype: int64

In [20]:
# missing values in Litter were preserved, as missingness may reflect
# unrecorded observations rather than true absence.
df["Litter"].value_counts(dropna=False)

Litter
Some        1323
None        1211
NaN          382
Abundant     102
Name: count, dtype: int64

In [21]:
# missing values in sky_condition were preserved because some free-text
# weather descriptions could not be confidently mapped to standard conditions.
df["sky_condition"].value_counts(dropna=False)

sky_condition
cloudy        962
sunny         734
NaN           520
overcast      348
clear         150
mist          121
rainy          78
fog            54
drizzle        28
sprinkling     18
wet             5
Name: count, dtype: int64

In [22]:
# to see why above_ground_numeric have missing values
# need to delete "Location", "Above Ground Sighter Measurement" in the drop_cols[] above

# df[df["above_ground_numeric"].isna()][
#     ["Location", "Above Ground Sighter Measurement"]
# ].head(20)

In [23]:
df["above_ground_numeric"].describe()

count    2904.000000
mean        4.153581
std        10.559927
min         0.000000
25%         0.000000
50%         0.000000
75%         2.000000
max       180.000000
Name: above_ground_numeric, dtype: float64

In [24]:
# # check leakage columns removed
# leakage_cols = ["Approaches", "Indifferent", "Runs from"]
# [c for c in leakage_cols if c in df.columns]

In [25]:
# check raw replaced columns removed
raw_cols = [
    "Location",
    "Above Ground Sighter Measurement",
    "Highlight Fur Color",
    "Other Animal Sightings",
    "Sighter Observed Weather Data"
]

[c for c in raw_cols if c in df.columns]

['Sighter Observed Weather Data']

In [26]:
A = df["Approaches"]
I = df["Indifferent"]
R = df["Runs from"]


conditions = [
    (A & ~I & ~R),
    (A & I & ~R),
    (~A & ~I & R),
    (~A & I & R)
]

choices = [1, 1, 0, 0]

df["y"] = np.select(conditions, choices, default=np.nan)

# df = df.dropna(subset=["y"])


# print(df["y"].value_counts(dropna=False))
# print(df.shape)

In [27]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3018 entries, 0 to 3022
Data columns (total 48 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   X                              3018 non-null   float64       
 1   Y                              3018 non-null   float64       
 2   Unique Squirrel ID             3018 non-null   object        
 3   Hectare                        3018 non-null   object        
 4   Shift                          3018 non-null   object        
 5   Date                           3018 non-null   datetime64[ns]
 6   Age                            3018 non-null   object        
 7   Primary Fur Color              3018 non-null   object        
 8   Running                        3018 non-null   bool          
 9   Chasing                        3018 non-null   bool          
 10  Climbing                       3018 non-null   bool          
 11  Eating                

In [28]:
df.to_csv("cleaned_table_2.csv", index=False)
df.to_parquet("data/cleaned_table_2.parquet", index=False)